# 6.3 预训练 CNN 迁移学习：PANNs CNN14

基于 PANNs CNN14（AudioSet 预训练）在 CTMP 6 类数据集上做迁移学习，
对比三种迁移策略：Feature Extraction、Full Fine-tuning、Layer-wise LR。

评估协议（与 6.2 一致）：
- **选择**：train split 训练 → val split 调度并选择最佳轮次
- **内部**：锁定权重后在 test split 做最终评估
- **外部**：同一模型直接在 external_test（ChMusic）上推理
- **划分敏感性**：在 3 份冻结的 train/val/test 划分上评估，取均值和总体标准差；external_test固定不变


## 1. 环境准备


In [ ]:
import sys
from pathlib import Path

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录；PROJECT_ROOT 指向 CODE/
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
PROJECT_ROOT = _p / "CODE"  # CODE/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = [
    'Hiragino Sans GB', 'PingFang SC', 'Arial Unicode MS', 'STHeiti', 'Heiti TC',
    'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans',
]
matplotlib.rcParams['axes.unicode_minus'] = False
print('matplotlib:', matplotlib.__version__)


### 1.1 依赖与数据齐备性检查

`torchlibrosa` 提供CNN14内部的mel前处理层，是本节额外使用的依赖。
PANNs CNN14预训练权重（327,428,481字节）首次运行时由`download_cnn14_weights()`获取，
不在这里检查。


In [ ]:
from chapter06._common import check_environment

check_environment(notebook='06_3', require_ctmp=True)


In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

from chapter06._common import (
    add_recall_colorbar,
    get_device,
    plot_confusion_matrix,
    setup_chinese_font,
)
from chapter06._common.ctmp_loader import (
    CTMP_CLASSES,
    build_label_map,
    get_ctmp_output_dir,
    load_ctmp_segments,
)
from chapter06.pretrained_cnn import (
    TransferCnn14,
    TrainConfig,
    run_experiment,
)

setup_chinese_font()

OUT_DIR = PROJECT_ROOT / 'chapter06' / 'pretrained_cnn' / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [0, 1, 2]
LABEL_MAP = build_label_map()
CLASS_NAMES = list(CTMP_CLASSES)
N_CLASSES = len(CLASS_NAMES)

print(f'CTMP 类别: {CLASS_NAMES} ({N_CLASSES} 类)')
print(f'CTMP 输出目录: {get_ctmp_output_dir()}')
print(f'device: {get_device()}')
print(f'torch: {torch.__version__}')


## 2. PANNs CNN14 架构介绍

CNN14 是 PANNs（Pretrained Audio Neural Networks）发布的模型变体之一：

- 预训练数据：AudioSet的527类弱标注音频；论文报告当时可下载训练片段为1,934,187条
- 参数量：约8100万；论文表格与本地完整checkpoint结构的统计口径略有差异
- 结构：Mel前端 + 6个ConvBlock（通道64→128→256→512→1024→2048）+ 频率均值与时间Max/Avg汇聚 + `fc1`
- 输出：2048维embedding；原AudioSet头将527维logits经sigmoid转换为类别概率

迁移到下游任务时，新建`Linear(2048, n_classes)`分类头；原527类AudioSet头保留在
checkpoint结构中，但不参与本章下游前向计算，并在全量解冻策略中保持冻结。


In [ ]:
from chapter06.pretrained_cnn.model import download_cnn14_weights
from chapter06.pretrained_cnn import fine_tune_all

# 确保权重已下载
ckpt_path = download_cnn14_weights()
print(f'CNN14 权重路径: {ckpt_path}')

# 实例化模型（frozen backbone）
model_frozen = TransferCnn14(n_classes=N_CLASSES, freeze_base=True)
model_frozen.load_from_pretrain(ckpt_path)
print(f'Feature Extraction 可训练参数: {model_frozen.num_params():,}')

# 实例化模型（unfrozen）
model_full = TransferCnn14(n_classes=N_CLASSES, freeze_base=False)
model_full.load_from_pretrain(ckpt_path)
fine_tune_all(model_full)
print(f'Full Fine-tune 可训练参数: {model_full.num_params():,}')

del model_frozen, model_full


In [ ]:
# 架构对比表
comparison = pd.DataFrame([
    {'模型': 'SimpleAudioCNN (6.2)', '参数量': '93,670', '预训练': '无', '输入': 'log-mel谱（64 bins）'},
    {'模型': 'CNN14 (6.3)', '参数量': '约8100万（口径见正文）', '预训练': 'AudioSet 527 类', '输入': '原始波形 (32 kHz)'},
])
print(comparison.to_string(index=False))


## 3. 三种迁移模式

| 模式 | 策略 | 可训练参数 |
|------|------|-----------|
| Feature Extraction | 冻结 backbone，只训练 Linear(2048, 6) | 12,294 |
| Full Fine-tuning | 解冻下游前向路径，统一 lr=1e-5 | 80,769,542 |
| Layer-wise LR（骨干/head两组） | backbone lr=1e-5，head lr=1e-3 | 80,769,542 |

三种模式改变了可训练参数范围、学习率设置和骨干网络的训练态。参数更新范围扩大后，
需要保存的梯度与优化器状态增多，结果也可能对当前小规模数据和优化配置更敏感。
三种完整策略在 CTMP 上的相对结果由后续实验给出。


## 4. 训练与评估（三份冻结划分 × 3 模式）

对每种迁移模式评估 3 份冻结划分。代码接口中的 `seed=0/1/2` 是三份
manifest 的编号；每次只用 val 调度和选择权重，锁定模型后记录内部 test
和外部 external_test 的指标。


In [ ]:
from chapter06.pretrained_cnn.model import download_cnn14_weights

# 确保权重可用
_ = download_cnn14_weights()

MODE_NAMES = {
    'feature_extraction': 'Feature Extraction',
    'fine_tune': 'Full Fine-tune',
    'layer_wise': 'Layer-wise LR',
}

train_cfg = TrainConfig(
    epochs=30, batch_size=16, lr=1e-3, backbone_lr=1e-5,
    weight_decay=1e-4, num_workers=0, log_every=10,
    label_smoothing=0.1, scheduler_patience=5, scheduler_factor=0.5,
)

all_results = []
RUN_CACHE = OUT_DIR / 'run_cache' / 'cnn14_val'

for mode in ['feature_extraction', 'fine_tune', 'layer_wise']:
    print(f'\n{"="*60}')
    print(f'  模式: {MODE_NAMES[mode]}')
    print(f'{"="*60}')
    for seed in SEEDS:
        t0 = time.perf_counter()
        result = run_experiment(
            seed=seed, mode=mode, cfg=train_cfg,
            tag=f'{MODE_NAMES[mode]}-split{seed}',
            cache_dir=RUN_CACHE,
        )
        all_results.append(result)
        elapsed = time.perf_counter() - t0
        print(f'  [{MODE_NAMES[mode]}] split {seed} 完成 | '
              f'test_acc={result["test_acc"]:.3f} ext_acc={result["ext_acc"]:.3f} | '
              f'耗时 {elapsed:.1f}s')
    print()

print(f'\n全部完成：{len(all_results)} 次实验')


## 5. 主结果表：3 份冻结划分的均值 ± 总体标准差

内部（test）与外部（external_test）的准确率和 macro-F1 并列，Gap = 内部 − 外部。


In [ ]:
summary_rows = []
for mode in ['feature_extraction', 'fine_tune', 'layer_wise']:
    sub = [r for r in all_results if r['mode'] == mode]
    test_accs = [r['test_acc'] for r in sub]
    test_f1s = [r['test_f1'] for r in sub]
    ext_accs = [r['ext_acc'] for r in sub]
    ext_f1s = [r['ext_f1'] for r in sub]
    summary_rows.append({
        '方法': f'CNN14 · {MODE_NAMES[mode]}',
        '内部 Acc': f"{np.mean(test_accs):.3f} ± {np.std(test_accs):.3f}",
        '内部 F1': f"{np.mean(test_f1s):.3f} ± {np.std(test_f1s):.3f}",
        '外部 Acc': f"{np.mean(ext_accs):.3f} ± {np.std(ext_accs):.3f}",
        '外部 F1': f"{np.mean(ext_f1s):.3f} ± {np.std(ext_f1s):.3f}",
        'Gap (Acc)': f"{np.mean(test_accs) - np.mean(ext_accs):+.3f}",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'method_comparison.csv', index=False)
print('written:', (OUT_DIR / 'method_comparison.csv').resolve())
df_summary


## 6. 训练曲线对比（冻结划分0，3 模式）

取冻结划分0的训练历史，画3×2网格：每行一种模式，左列loss，右列val指标。


In [ ]:
def plot_history(history, title, ax_loss, ax_metric):
    epochs = range(1, len(history['train_loss']) + 1)
    ax_loss.plot(epochs, history['train_loss'], label='train', color='0.3')
    ax_loss.plot(epochs, history['val_loss'], label='val', color='0.6', linestyle='--')
    ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('loss')
    ax_loss.set_title(f'{title} · loss'); ax_loss.legend()
    ax_metric.plot(epochs, history['val_acc'], label='accuracy', color='0.3')
    ax_metric.plot(epochs, history['val_f1'], label='macro-F1', color='0.6', linestyle='--')
    ax_metric.set_xlabel('epoch'); ax_metric.set_ylabel('指标')
    ax_metric.set_title(f'{title} · val 指标'); ax_metric.legend()

seed0_results = [r for r in all_results if r['seed'] == 0]

fig, axes = plt.subplots(3, 2, figsize=(12, 12))
for row, r in enumerate(seed0_results):
    plot_history(r['history'], MODE_NAMES[r['mode']], axes[row, 0], axes[row, 1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'training_curves.png', dpi=600, bbox_inches='tight')
plt.show()


## 7. 混淆矩阵：内部 vs 外部（划分 0，按验证集选择模式）

先按 3 份划分的平均 val macro-F1 选择模式，并以平均 val Acc 处理并列；
若两项仍相同，则按表中声明顺序选择。随后绘制该模式在划分 0 上的内部 test
和外部 external_test 混淆矩阵。最终测试集不参与模式选择。


In [ ]:
mean_val_f1 = {
    mode: np.mean([r['val_f1'] for r in all_results if r['mode'] == mode])
    for mode in MODE_NAMES
}
mean_val_acc = {
    mode: np.mean([r['val_acc'] for r in all_results if r['mode'] == mode])
    for mode in MODE_NAMES
}
mode_order = {mode: idx for idx, mode in enumerate(MODE_NAMES)}
selected_mode = sorted(
    MODE_NAMES,
    key=lambda mode: (-mean_val_f1[mode], -mean_val_acc[mode], mode_order[mode]),
)[0]
best = next(r for r in seed0_results if r['mode'] == selected_mode)
print(f'按平均 val macro-F1 选择: {MODE_NAMES[selected_mode]} | '
      f'val_f1={mean_val_f1[selected_mode]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion_matrix(
    axes[0], best['y_test'], best['pred_test'],
    class_names=CLASS_NAMES, title=f'{MODE_NAMES[best["mode"]]} — 内部 test',
    labels=list(range(N_CLASSES)),
)
plot_confusion_matrix(
    axes[1], best['y_ext'], best['pred_ext'],
    class_names=CLASS_NAMES, title=f'{MODE_NAMES[best["mode"]]} — 外部 external_test',
    labels=list(range(N_CLASSES)),
)
fig.subplots_adjust(wspace=0.05)
add_recall_colorbar(fig, [axes[0], axes[1]])
fig.savefig(FIG_DIR / 'confusion_internal_vs_external.png', dpi=600, bbox_inches='tight')
plt.show()


## 8. 与 6.1/6.2 全量对比

读取 6.2 simple_cnn 的 `full_comparison.csv`，与本节 CNN14 结果并列。


In [ ]:
# 读 6.2 结果（已包含 6.1b）
cnn_csv = PROJECT_ROOT / 'chapter06' / 'simple_cnn' / 'outputs' / 'full_comparison.csv'
if not cnn_csv.exists():
    raise FileNotFoundError(
        '缺少6.2结果：请先执行06_2_simple_cnn.ipynb，再生成全章比较表'
    )
df_prev = pd.read_csv(cnn_csv, dtype=str)
df_all = pd.concat([df_prev, df_summary], ignore_index=True)

df_all.to_csv(OUT_DIR / 'full_comparison.csv', index=False)
print('written:', (OUT_DIR / 'full_comparison.csv').resolve())
df_all


## 9. 结论

本节在同一训练轮数和三份冻结划分上比较三种参数更新策略。结果表用于回答：

- 冻结骨干网络时，固定AudioSet表示配合新线性头能达到什么结果；
- 解冻下游前向路径后，验证集选择的权重是否改善内部或外部指标；
- backbone与head使用不同学习率时，结果是否位于两种端点策略之间。

**下一步（6.4）**：比较AST的Feature Extraction、LoRA和全量微调，并考察
CLAP零样本分类。
